# Gapfinder Prototype

This notebook prototypes the core flow for the Gapfinder application.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
from IPython.display import display, Markdown

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"


## Step 1: Extract & Structure Knowledge
Fetch the transcript and extract key concepts.

In [ ]:
def get_video_id(url):
    """Extract video ID from YouTube URL"""
    if "v=" in url:
        return url.split("v=")[1][:11]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1][:11]
    return url

def get_transcript(video_id):
    """Download transcript"""
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
    return " ".join([t["text"] for t in transcript_list])

def extract_concepts(transcript_text):
    """Use LLM to extract key concepts from the transcript"""
    prompt = f"""
    You are an expert educator. Extract the most important key concepts from the following transcript of an educational video.
    Return a bulleted list of 3-5 core concepts with a brief explanation for each.
    
    Transcript:
    {transcript_text}
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

# Test Step 1
video_url = "https://www.youtube.com/watch?v=njX2bu-_Vw4" # Example: 3Blue1Brown Neural Networks
video_id = get_video_id(video_url)
print(f"Fetching transcript for video ID: {video_id}")
transcript = get_transcript(video_id)
print(f"Transcript length: {len(transcript)} characters")

print("\nExtracting Concepts...")
concepts = extract_concepts(transcript)
display(Markdown(concepts))


## Step 2: Generate Diagnostic Questions
Generate questions based on the concepts.

In [ ]:
def generate_questions(concepts):
    """Generate diagnostic questions based on concepts"""
    prompt = f"""
    Based on the following key concepts from a video, generate 3 diagnostic questions. 
    The questions should not be simple recall questions. They should be:
    1. A concept coverage question (testing understanding of the core mechanism).
    2. An "explain in your own words" prompt.
    3. An application question (transfer knowledge to a new scenario).
    
    Concepts:
    {concepts}
    
    Output the questions clearly numbered.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

print("Generating Questions...")
questions = generate_questions(concepts)
display(Markdown(questions))


## Step 3 & 4: User Answers & Gap Detection
Provide mock answers and detect gaps.

In [ ]:
# Step 3: Mock User Answers
mock_answers = """
1. A neural network is a machine learning model that takes some inputs, does some math, and gives an output. The weights are like importance scores.
2. The activation function is how the network decides if a neuron should fire. I think it uses a sigmoid function to squash values between 0 and 1.
3. If I change the image, the network will probably just output random numbers because it has not been trained on that specific image.
"""
display(Markdown(f"**User Answers:**\n{mock_answers}"))


In [ ]:
# Step 4: Gap Detection
def detect_gaps(concepts, questions, answers):
    prompt = f"""
    You are an expert tutor. Evaluate the user's answers to the diagnostic questions based on the core concepts of the video.
    
    Core Concepts:
    {concepts}
    
    Questions:
    {questions}
    
    User's Answers:
    {answers}
    
    Provide a structured report including:
    - **What they understood well:** Identify correct points in their answers.
    - **What they misunderstood or missed:** Clearly highlight the knowledge gaps or misconceptions.
    - **What to revisit:** Specifically state which concepts they need to review.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

print("Detecting Gaps...")
gap_report = detect_gaps(concepts, questions, mock_answers)
display(Markdown(gap_report))
